# Process Simulation Starter

**Task:** [Replace with task title]
**Date:** [YYYY-MM-DD]

Standard template for separation, compression, and heat exchange process simulations.

In [ ]:
# ── NeqSim Setup (dual-boot: devtools or pip) ──
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

import matplotlib.pyplot as plt
import numpy as np
import json, os, pathlib

# ── Resolve paths ──
NOTEBOOK_DIR = pathlib.Path(globals().get(
    "__vsc_ipynb_file__", os.path.abspath("step2_analysis/01_process_sim.ipynb")
)).resolve().parent
TASK_DIR = NOTEBOOK_DIR.parent
FIGURES_DIR = TASK_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
# ── Import NeqSim classes ──
if NEQSIM_MODE == "pip":
    from neqsim import jneqsim
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ProcessSystem = jneqsim.process.processmodel.ProcessSystem
    Stream = jneqsim.process.equipment.stream.Stream
    Separator = jneqsim.process.equipment.separator.Separator
    ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
    Compressor = jneqsim.process.equipment.compressor.Compressor
    Cooler = jneqsim.process.equipment.heatexchanger.Cooler
    Heater = jneqsim.process.equipment.heatexchanger.Heater
    HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
    ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
    Mixer = jneqsim.process.equipment.mixer.Mixer
    Splitter = jneqsim.process.equipment.splitter.Splitter
    Recycle = jneqsim.process.equipment.util.Recycle
    print("Classes imported from jneqsim")

## 1. Fluid Definition

Define the thermodynamic system. Choose the appropriate EOS:
- **SRK**: General hydrocarbon systems
- **PR**: Heavier oils, better liquid density
- **SRK-CPA**: Systems with water, methanol, MEG (hydrogen bonding)

**Remember:** Temperature in Kelvin, pressure in bara for constructors.

In [ ]:
# ── Create fluid ──
fluid = SystemSrkEos(273.15 + 25.0, 60.0)  # 25 degC, 60 bara

# Add components (name, mole fraction)
fluid.addComponent("nitrogen", 0.01)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("methane", 0.80)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.05)
fluid.addComponent("i-butane", 0.01)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("n-pentane", 0.01)

# MANDATORY: set mixing rule
fluid.setMixingRule("classic")

print("Fluid created: {} components".format(fluid.getNumberOfComponents()))

## 2. Process Configuration

In [ ]:
# ── Build process ──
process = ProcessSystem()

# Feed stream
feed = Stream("Feed", fluid)
feed.setFlowRate(50000.0, "kg/hr")
feed.setTemperature(30.0, "C")
feed.setPressure(60.0, "bara")
process.add(feed)

# HP Separator
hp_sep = Separator("HP Sep", feed)
process.add(hp_sep)

# Gas cooler
cooler = Cooler("Gas Cooler", hp_sep.getGasOutStream())
cooler.setOutTemperature(273.15 + 20.0)
process.add(cooler)

# Compressor
comp = Compressor("Compressor", cooler.getOutletStream())
comp.setOutletPressure(120.0, "bara")
process.add(comp)

# LP valve
lp_valve = ThrottlingValve("LP Valve", hp_sep.getLiquidOutStream())
lp_valve.setOutletPressure(10.0, "bara")
process.add(lp_valve)

# LP Separator
lp_sep = Separator("LP Sep", lp_valve.getOutletStream())
process.add(lp_sep)

print("Process built: {} units".format(process.size()))

In [ ]:
# ── Run simulation ──
process.run()
print("Simulation complete")

## 3. Results

In [ ]:
# ── Extract key results ──
results_data = {
    "HP Sep gas T (C)": hp_sep.getGasOutStream().getTemperature() - 273.15,
    "HP Sep gas P (bara)": hp_sep.getGasOutStream().getPressure(),
    "Compressor power (kW)": comp.getPower("kW"),
    "Compressor outlet T (C)": comp.getOutletStream().getTemperature() - 273.15,
    "LP Sep gas flow (kg/hr)": lp_sep.getGasOutStream().getFlowRate("kg/hr"),
    "LP Sep liquid flow (kg/hr)": lp_sep.getLiquidOutStream().getFlowRate("kg/hr"),
}

print("\n{:<35} {:>15}".format("Parameter", "Value"))
print("-" * 52)
for k, v in results_data.items():
    print("{:<35} {:>15.2f}".format(k, v))

## 4. Visualization

In [ ]:
# ── Example: Pressure-temperature profile ──
equipment_names = ["Feed", "HP Sep", "Gas Cooler", "Compressor"]
temps = [feed.getTemperature() - 273.15,
         hp_sep.getGasOutStream().getTemperature() - 273.15,
         cooler.getOutletStream().getTemperature() - 273.15,
         comp.getOutletStream().getTemperature() - 273.15]
pressures = [feed.getPressure(),
             hp_sep.getGasOutStream().getPressure(),
             cooler.getOutletStream().getPressure(),
             comp.getOutletStream().getPressure()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(equipment_names, temps, color='steelblue')
ax1.set_ylabel('Temperature (\u00b0C)')
ax1.set_title('Temperature Profile')
ax1.grid(axis='y', alpha=0.3)

ax2.bar(equipment_names, pressures, color='coral')
ax2.set_ylabel('Pressure (bara)')
ax2.set_title('Pressure Profile')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "process_profile.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to figures/process_profile.png")

### Discussion

**Observation:** [What the figure shows with specific numbers]

**Physical mechanism:** [Why the observed behavior occurs]

**Engineering implication:** [What it means for design/operations]

**Recommendation:** [Specific action to take]

## 5. Save Results

In [ ]:
# ── Save results.json ──
results_path = TASK_DIR / "results.json"
if results_path.exists():
    with open(results_path, "r") as f:
        results = json.load(f)
else:
    results = {}

results["key_results"] = results_data
results["validation"] = {
    "mass_balance_error_pct": 0.0,  # TODO: calculate
    "acceptance_criteria_met": True,
}
results["approach"] = "SRK EOS with classic mixing rule. [Edit as needed]"
results["conclusions"] = "[Replace with conclusions]"
results["figure_captions"] = {
    "process_profile.png": "Temperature and pressure profile across process equipment.",
}
results["figure_discussion"] = []

with open(str(results_path), "w") as f:
    json.dump(results, f, indent=2)

print("results.json saved to", results_path)